# 07. So sánh mô hình và trả lời ba câu hỏi nghiên cứu

1. LSTM và GRU có dự báo tốt hơn phương pháp truyền thống và học máy cơ bản không?
2. Dữ liệu vận hành Hủa Na có cải thiện dự báo không, biến nào có giá trị nhất?
3. Cách xử lý dữ liệu và huấn luyện ảnh hưởng thế nào, đặc biệt với lưu lượng lớn và mùa lũ?

Mọi so sánh dùng cùng ngày đích; bootstrap khối dịch chuyển (khối 30 ngày, 2000 lần) dùng **chung một bộ chỉ số** cho mọi cặp.
Mô hình / cấu hình được chọn theo **CV**; tập test chỉ để báo cáo.

In [ ]:
# ── Google Colab: gắn Google Drive, vào thư mục dự án, cài đúng phiên bản thư viện. Chạy trên máy thì tự bỏ qua. ──
import os, sys, glob, subprocess
DU_AN = '/content/drive/MyDrive/T_DAThS_Ninh'   # ← thư mục dự án trên Google Drive (notebook 01, mục 0 tạo ra)
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.isdir(os.path.join(DU_AN, 'src')):          # không thấy → tự tìm trong My Drive (sâu tối đa 3 cấp)
        hit = [h for k in range(1, 4) for h in glob.glob('/content/drive/MyDrive/' + '*/' * k + 'src/results_store.py')]
        assert hit, ('Chưa có dự án trên Google Drive: mở notebook 01_PhanTich_XuLy_DuLieu và chạy 2 ô của mục 0 '
                     '(lấy code từ GitHub, giải nén data/kết quả), hoặc sửa biến DU_AN ở đầu ô này')
        DU_AN = os.path.dirname(os.path.dirname(hit[0]))
    os.chdir(DU_AN)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements_colab.txt'], check=True)

import json, warnings
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / 'src' / 'results_store.py').exists():
    assert ROOT != ROOT.parent, 'Không tìm thấy thư mục dự án (thư mục chứa src/): mở notebook từ trong dự án'
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40); pd.set_option('display.max_colwidth', 80)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': .3})
from src.utils import load_config, results_dir, dataset_meta
cfg = load_config()
print('Thư mục dự án:', ROOT, '| kết quả lưu ở:', cfg['paths']['results'])
print('Thiết bị:', f'GPU {torch.cuda.get_device_name(0)}' if torch.cuda.is_available() else
      'CPU (trên Colab: Runtime → Change runtime type → GPU để chạy nhanh hơn)')
from src.evaluation import comparison as C
from src.evaluation.metrics import LABEL, DIGITS
from src.training.train import DISPLAY, MODEL_NAMES
from src import pipeline
from src import results_store as S
OUT = results_dir(cfg, 'comparison')
cv, met, pred = C.load_all(cfg, 'H1')
cv10, met10, pred10 = C.load_all(cfg, 'H10')
M7 = MODEL_NAMES; DL = ['rnn', 'lstm', 'gru']; BL = ['persistence', 'ridge', 'extra_trees', 'xgboost']
conf = {m: pipeline.load_conf(cfg, m) for m in M7}
display(pd.DataFrame({DISPLAY[m]: {'siêu tham số': c.get('hp'), 'trọng số E5': c.get('weights'), 'λ': c.get('lambda')} for m, c in conf.items()}).T)

## 1. Kết quả CV (t+1): NSE trung bình 3 fold

In [ ]:
cvm = C.cv_matrix(cv);
order = [c for c in ['PERS', 'E1', 'E2', 'E3', 'E4', 'P_F4', 'P_F5', 'P_EMA', 'W_g1_cnone', 'W_g2_cnone', 'W_g1_c20', 'W_g2_c20', 'E5', 'PEN_l0.5', 'PEN_l1.0', 'E6', 'E56', 'FUT'] if c in cvm.columns]
cvm = cvm[order]; display(cvm.round(3)); cvm.to_csv(OUT / 'cv_nse_H1.csv', encoding='utf-8-sig')
cvq = C.cv_matrix(cv, 'NSE Q>Q95 CV')[order]; print('NSE Q>Q95 (CV):'); display(cvq.round(3)); cvq.to_csv(OUT / 'cv_nse_q95_H1.csv', encoding='utf-8-sig')
pers = cvm.loc['Persistence', 'PERS']
FIN = ['E1', 'E2', 'E3', 'E4', 'P_F4', 'P_F5', 'P_EMA', 'E5', 'E6', 'E56']
best_cv = {m: cvm.loc[DISPLAY[m], [c for c in FIN if c in cvm.columns]].dropna().idxmax() for m in M7 if m != 'persistence'}
best_cv['persistence'] = 'PERS'
print('Cấu hình tốt nhất theo CV (không tính FUT):', {DISPLAY[m]: e for m, e in best_cv.items()})

## 2. Câu hỏi 1 — LSTM/GRU so với baseline (test khoá, t+1)

In [ ]:
KEYM = ['NSE', 'KGE', 'r2', 'RMSE', 'MAE', 'PBIAS', 'NSE_Q95', 'RMSE_Q95', 'Peak_RMSE', 'Peak_Bias', 'Peak_Timing']
met_p = met.copy(); met_p.loc[met_p.model == 'persistence', 'exp'] = 'E2'      # Persistence không dùng biến → đặt cạnh E2
t1 = C.test_table(met_p, ['E2'], KEYM); display(t1); t1.to_csv(OUT / 'test_E2_H1.csv', encoding='utf-8-sig')
rows = []
for m in M7:
    e = best_cv[m]; d = met[(met.model == m) & (met.exp == e)]
    rows.append({'mô hình': DISPLAY[m], 'cấu hình tốt nhất (CV)': e, 'NSE CV': cvm.loc[DISPLAY[m], e], **{LABEL[k]: C.fmt(d[k], DIGITS[k]) for k in KEYM}})
tb = pd.DataFrame(rows).set_index('mô hình'); display(tb); tb.to_csv(OUT / 'test_best_cv_H1.csv', encoding='utf-8-sig')
pairs = []
for a in ['lstm', 'gru']:
    for b in BL:
        pairs.append((a, 'E2', b, 'PERS' if b == 'persistence' else 'E2', f'{DISPLAY[a]} − {DISPLAY[b]} (E2)'))
    pairs.append((a, 'E2', 'rnn', 'E2', f'{DISPLAY[a]} − RNN (E2)'))
pairs += [('lstm', 'E2', 'gru', 'E2', 'LSTM − GRU (E2)'), ('lstm', 'E5', 'gru', 'E5', 'LSTM − GRU (E5)')]
for a in ['lstm', 'gru']:
    for b in ['ridge', 'xgboost']:
        pairs.append((a, best_cv[a], b, best_cv[b], f'{DISPLAY[a]} {best_cv[a]} − {DISPLAY[b]} {best_cv[b]} (tốt nhất theo CV)'))
bq1 = C.bootstrap_table(cfg, pred, pairs); display(bq1); bq1.to_csv(OUT / 'bootstrap_cau1.csv', index=False, encoding='utf-8-sig')

## 3. Câu hỏi 2 — vai trò dữ liệu Hủa Na

In [ ]:
hn = cvm[['E1', 'E2', 'E3', 'E4', 'P_F4', 'P_F5']].drop('Persistence')
print('NSE CV:'); display(hn.round(3))
print('Δ so với E1 (CV):'); display(hn.sub(hn['E1'], axis=0).drop(columns='E1').round(3))
pairs = [(m, e, m, 'E1', f'{DISPLAY[m]}: {e} − E1') for m in M7 if m != 'persistence' for e in ['E2', 'E3', 'E4']]
bq2 = C.bootstrap_table(cfg, pred, pairs); display(bq2); bq2.to_csv(OUT / 'bootstrap_cau2_huana.csv', index=False, encoding='utf-8-sig')
t2 = C.test_table(met, ['E1', 'E2', 'E3', 'E4'], ['NSE', 'KGE', 'NSE_Q95', 'Peak_Bias']); t2.to_csv(OUT / 'test_huana_H1.csv', encoding='utf-8-sig')
piv = met[met.exp.isin(['E1', 'E2', 'E3', 'E4', 'P_F4', 'P_F5'])].groupby(['model', 'exp']).NSE.mean().unstack()
piv = piv.reindex([m for m in M7 if m in piv.index]); piv.index = [DISPLAY[m] for m in piv.index]; print('NSE test:'); display(piv.round(3))
fig, ax = plt.subplots(figsize=(10, 3.8)); xs = np.arange(len(piv)); w = .2
for j, e in enumerate(['E1', 'E2', 'E3', 'E4']):
    ax.bar(xs + (j - 1.5) * w, cvm.loc[piv.index, e], w, label=f'{e} (CV)')
ax.set_xticks(xs); ax.set_xticklabels(piv.index); ax.set_ylabel('NSE CV'); ax.legend(fontsize=8, ncol=4); ax.set_ylim(0, None)
ax.set_title('Vai trò dữ liệu Hủa Na (CV, t+1)'); plt.tight_layout(); plt.savefig(OUT / 'fig_huana_cv.png'); plt.show()

## 4. Câu hỏi 3 — xử lý dữ liệu và huấn luyện

### 4.1 Ngưỡng khoảng thiếu (từ notebook 02)

In [ ]:
gs = pd.read_csv(results_dir(cfg, 'dataset') / 'chon_nguong_khoang_thieu.csv'); display(gs.round(3))
print('G chọn =', dataset_meta(cfg)['max_gap'])

### 4.2 Hàm mất mát có trọng số (E5), phạt dự báo thiếu (E6, E56), biến làm trơn (P_EMA), mưa tương lai (FUT)

In [ ]:
print('Δ NSE CV so với E2:'); display(cvm.drop('Persistence')[[c for c in ['E5', 'E6', 'E56', 'P_EMA', 'FUT'] if c in cvm]].sub(cvm.drop('Persistence')['E2'], axis=0).round(3))
print('Δ NSE Q>Q95 CV so với E2:'); display(cvq.drop('Persistence')[[c for c in ['E5', 'E6', 'E56', 'P_EMA', 'FUT'] if c in cvq]].sub(cvq.drop('Persistence')['E2'], axis=0).round(3))
pairs = [(m, 'E5', m, 'E2', f'{DISPLAY[m]}: E5 − E2') for m in M7 if m != 'persistence']
pairs += [(m, e, m, 'E2', f'{DISPLAY[m]}: {e} − E2') for m in ['lstm', 'gru'] for e in ['E6', 'E56']]
pairs += [(m, 'P_EMA', m, 'E2', f'{DISPLAY[m]}: P_EMA − E2') for m in M7 if m != 'persistence']
pairs += [(m, 'FUT', m, 'E2', f'{DISPLAY[m]}: FUT − E2 (cận trên)') for m in M7 if m != 'persistence']
bq3 = C.bootstrap_table(cfg, pred, pairs); display(bq3); bq3.to_csv(OUT / 'bootstrap_cau3.csv', index=False, encoding='utf-8-sig')
t3 = C.test_table(met, ['E2', 'E5', 'E6', 'E56'], ['NSE', 'KGE', 'PBIAS', 'UNDER', 'NSE_Q95', 'RMSE_Q95', 'UNDER_Q95', 'Peak_RMSE', 'Peak_Bias', 'Peak_Timing'])
display(t3); t3.to_csv(OUT / 'test_luu_luong_cao_H1.csv', encoding='utf-8-sig')

### 4.3 Mùa lũ 2025 (t+1, trung bình các hạt giống)

In [ ]:
obs = pd.Series(dtype=float)
fig, axs = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
a, b = pd.Timestamp('2025-07-01'), pd.Timestamp('2025-10-31')
for ax, e in zip(axs, ['E2', 'E5']):
    first = True
    for m, c in [('persistence', '0.6'), ('ridge', 'C2'), ('xgboost', 'C1'), ('lstm', 'C3'), ('gru', 'C0')]:
        ee = 'PERS' if m == 'persistence' else e
        r = C.arrays(pred, m, ee)
        if r is None: continue
        O, M_, P, idx = r; s = pd.Series(np.mean(P, axis=0)[:, 0], index=idx + pd.Timedelta(days=1))
        if first: ax.plot(pd.Series(O[:, 0], index=idx + pd.Timedelta(days=1)).loc[a:b], 'k', lw=1.3, label='quan trắc'); first = False
        ax.plot(s.loc[a:b], color=c, lw=1, label=DISPLAY[m])
    ax.set_title(f'{e}: {pipeline.EXP_NAME[e]}'); ax.set_ylabel('Q đến CĐ (m³/s)'); ax.legend(fontsize=8, ncol=6)
plt.tight_layout(); plt.savefig(OUT / 'fig_mua_lu_2025_H1.png'); plt.show()

## 5. Bài toán mở rộng t+1 … t+10

In [ ]:
met10_p = met10.copy(); met10_p.loc[met10_p.model == 'persistence', 'exp'] = 'E2'
t10 = C.test_table(met10_p, ['E1', 'E2', 'E5'], ['NSE', 'KGE', 'RMSE', 'PBIAS', 'NSE_Q95', 'Peak_Bias'])
display(t10); t10.to_csv(OUT / 'test_H10.csv', encoding='utf-8-sig')
from src.evaluation.metrics import per_horizon
rows = []
for m in M7:
    for e in (['PERS'] if m == 'persistence' else ['E2', 'E5']):
        r = C.arrays(pred10, m, e)
        if r is None: continue
        O, M_, P, _ = r
        ph = pd.concat([per_horizon(O, p, M_)[['MAE', 'NSE']] for p in P]).groupby(level=0).mean()
        for h, x in ph.iterrows(): rows.append({'mô hình': DISPLAY[m], 'thực nghiệm': e, 'h': h, **x.to_dict()})
ph = pd.DataFrame(rows); ph.to_csv(OUT / 'theo_h_H10.csv', index=False, encoding='utf-8-sig')
fig, axs = plt.subplots(1, 2, figsize=(13, 4))
for (m, e), d in ph.groupby(['mô hình', 'thực nghiệm'], sort=False):
    ls = '-' if e != 'E5' else '--'
    for ax, k in zip(axs, ['MAE', 'NSE']): ax.plot(d.h, d[k], ls, marker='o', ms=3, label=f'{m} {e}')
axs[0].set_ylabel('MAE test (m³/s)'); axs[1].set_ylabel('NSE test'); axs[1].legend(fontsize=7, ncol=2)
for ax in axs: ax.set_xlabel('h (ngày)'); ax.set_xticks(range(1, 11))
plt.tight_layout(); plt.savefig(OUT / 'fig_theo_h_H10.png'); plt.show()
pairs = [(a, 'E2', b, 'PERS' if b == 'persistence' else 'E2', f'{DISPLAY[a]} − {DISPLAY[b]} (E2, 10 ngày)') for a in ['lstm', 'gru'] for b in BL + ['rnn']]
pairs += [('lstm', 'E2', 'gru', 'E2', 'LSTM − GRU (E2, 10 ngày)')] + [(m, 'E2', m, 'E1', f'{DISPLAY[m]}: E2 − E1 (10 ngày)') for m in M7 if m != 'persistence']
pairs += [(m, 'E5', m, 'E2', f'{DISPLAY[m]}: E5 − E2 (10 ngày)') for m in M7 if m != 'persistence']
b10 = C.bootstrap_table(cfg, pred10, pairs); display(b10); b10.to_csv(OUT / 'bootstrap_H10.csv', index=False, encoding='utf-8-sig')

## 6. Thí nghiệm A (L = 3 chung) và thí nghiệm B (L chọn theo CV)

**A — so sánh có kiểm soát:** mọi mô hình dùng L = 3 (siêu tham số khác dò lại với L cố định). **B — cấu hình tối ưu:** L ∈ {3, 7,
14, 30} chọn theo CV cho từng mô hình. Mô hình nào ở B đã chọn L = 3 thì A ≡ B (quy trình tất định, không chạy lại).

In [ ]:
cvA, metA, predA = C.load_all(cfg, 'H1', variant='L3')
L_of = {m: (pipeline.load_conf(cfg, m, 'L3') if not pipeline.load_conf(cfg, m, 'L3').get('same_as_base') else conf[m])['hp'].get('L') for m in M7}
print('L dùng ở thí nghiệm A:', {DISPLAY[m]: L_of[m] for m in M7}, '| ở thí nghiệm B:', {DISPLAY[m]: conf[m]['hp'].get('L') for m in M7})
mA = metA.copy(); mA.loc[mA.model == 'persistence', 'exp'] = 'E2'
tA = C.test_table(mA, ['E1', 'E2', 'E3', 'E4'], ['NSE', 'KGE', 'RMSE', 'MAE', 'PBIAS'])
print('Thí nghiệm A — test (t+1):'); display(tA); tA.to_csv(OUT / 'thi_nghiem_A_L3_test.csv', encoding='utf-8-sig')
cvmA = C.cv_matrix(cvA)[[c for c in ['PERS', 'E1', 'E2', 'E3', 'E4', 'E5'] if c in C.cv_matrix(cvA).columns]]
print('Thí nghiệm A — NSE CV:'); display(cvmA.round(3)); cvmA.to_csv(OUT / 'thi_nghiem_A_L3_cv.csv', encoding='utf-8-sig')
pairs = [(a, 'E2', b, 'PERS' if b == 'persistence' else 'E2', f'[A, L=3] {DISPLAY[a]} − {DISPLAY[b]} (E2)') for a in ['lstm', 'gru'] for b in BL + ['rnn']]
bA = C.bootstrap_multi(cfg, predA, pairs); display(bA); bA.to_csv(OUT / 'thi_nghiem_A_bootstrap.csv', encoding='utf-8-sig')

## 7. Kiểm định chéo theo từng fold và quy tắc một sai số chuẩn

Chỉ tiêu từng fold (trung bình hạt giống), trung bình và độ lệch chuẩn giữa 3 fold, thời gian huấn luyện một lần, số vòng.
**Quy tắc 1-SE** (khai báo trước, chỉ là phân tích phụ — lựa chọn chính vẫn theo NSE CV cao nhất): trong các cấu hình có
NSE CV ≥ NSE tốt nhất − SE (SE = ĐLC giữa 3 fold / √3), chọn cấu hình đơn giản nhất (L nhỏ, ít tham số / α lớn).

In [ ]:
ft = C.fold_table(cv, ['PERS', 'E1', 'E2', 'E3', 'E4', 'E5', 'E6', 'E56'])
display(ft.round(3)); ft.to_csv(OUT / 'cv_theo_fold_H1.csv', encoding='utf-8-sig')
rows = []
SIMPLE = {'ridge': (['L', 'alpha'], [True, False]), 'extra_trees': (['L', 'min_samples_leaf'], [True, False]),
          'xgboost': (['L', 'max_depth'], [True, True]), 'rnn': (['L', 'layers', 'hidden'], [True, True, True]),
          'lstm': (['L', 'layers', 'hidden'], [True, True, True]), 'gru': (['L', 'layers', 'hidden'], [True, True, True])}
for m, (keys, asc) in SIMPLE.items():
    tab = S.read_tuning(cfg, m)
    best = tab.iloc[0]; se = best['NSE CV (ĐLC giữa fold)'] / np.sqrt(3)
    pick = tab[tab['NSE CV'] >= best['NSE CV'] - se].sort_values(keys, ascending=asc).iloc[0]
    rows.append({'mô hình': DISPLAY[m], 'chọn (NSE CV cao nhất)': {k: best[k] for k in keys}, 'NSE CV': best['NSE CV'], 'SE': se,
                 'chọn theo 1-SE': {k: pick[k] for k in keys}, 'NSE CV (1-SE)': pick['NSE CV'], 'trùng?': all(pick[k] == best[k] for k in keys)})
ose = pd.DataFrame(rows).set_index('mô hình'); display(ose); ose.to_csv(OUT / 'quy_tac_1se.csv', encoding='utf-8-sig')

## 8. Bootstrap nhiều chỉ tiêu (ΔNSE, ΔKGE, ΔMAE, ΔRMSE, tỉ lệ A tốt hơn B)

In [ ]:
pairs = [(a, 'E2', b, 'PERS' if b == 'persistence' else 'E2', f'{DISPLAY[a]} − {DISPLAY[b]} (E2)') for a in ['lstm', 'gru'] for b in BL + ['rnn']]
pairs.append(('lstm', 'E2', 'gru', 'E2', 'LSTM − GRU (E2)'))
b1 = C.bootstrap_multi(cfg, pred, pairs); display(b1); b1.to_csv(OUT / 'bootstrap_nhieu_chi_tieu_cau1.csv', encoding='utf-8-sig')
pairs = [(m, e, m, 'E1', f'{DISPLAY[m]}: {e} − E1') for m in M7 if m != 'persistence' for e in ['E2', 'E3', 'E4']]
b2 = C.bootstrap_multi(cfg, pred, pairs); display(b2); b2.to_csv(OUT / 'bootstrap_nhieu_chi_tieu_huana.csv', encoding='utf-8-sig')
pairs = [(m, 'E5', m, 'E2', f'{DISPLAY[m]}: E5 − E2') for m in M7 if m != 'persistence'] + \
        [(m, e, m, 'E2', f'{DISPLAY[m]}: {e} − E2') for m in ['lstm', 'gru'] for e in ['E6', 'E56']]
b3 = C.bootstrap_multi(cfg, pred, pairs); display(b3); b3.to_csv(OUT / 'bootstrap_nhieu_chi_tieu_ham_mat_mat.csv', encoding='utf-8-sig')
print('Δ theo fold (CV) của E2/E3/E4 so với E1 — phụ thuộc giai đoạn thủy văn:')
g = cv[cv.exp.isin(['E1', 'E2', 'E3', 'E4'])].groupby(['model', 'exp', 'fold']).NSE.mean().unstack('exp')
dfold = g[['E2', 'E3', 'E4']].sub(g['E1'], axis=0).round(3); dfold.index = dfold.index.set_levels([DISPLAY[m] for m in dfold.index.levels[0]], level=0)
display(dfold.unstack('fold')); dfold.to_csv(OUT / 'huana_delta_theo_fold.csv', encoding='utf-8-sig')

## 9. Độ ổn định giữa hạt giống, đỉnh lũ từng sự kiện, độ nhạy với 102 ngày răng cưa chưa rõ nguyên nhân

Ba đỉnh test là **mô tả**, không phải bằng chứng thống kê. Độ nhạy: bỏ các ngày đích bị cờ "nghi nhiễu tính toán" (tên cờ cũ; Q đến CĐ
đã được xác nhận là số liệu đo) **khi chấm điểm** (nhãn và huấn luyện giữ nguyên) — trên test và trên val của CV.

In [ ]:
st = met[met.exp.isin(['E2', 'E5', 'E6', 'E56'])].groupby(['model', 'exp']).agg(NSE_TB=('NSE', 'mean'), NSE_ĐLC=('NSE', 'std'),
        NSE_min=('NSE', 'min'), NSE_max=('NSE', 'max'), số_hạt_giống=('NSE', 'size')).round(3)
st.index = st.index.set_levels([DISPLAY[m] for m in st.index.levels[0]], level=0); display(st); st.to_csv(OUT / 'on_dinh_hat_giong.csv', encoding='utf-8-sig')
peaks = [pd.Timestamp(p) for p in dataset_meta(cfg)['test_peaks']]
pk = C.peak_table(pred, peaks, [('persistence', 'PERS')] + [(m, e) for m in M7 if m != 'persistence' for e in ['E2', 'E5']])
display(pk.round(1)); pk.to_csv(OUT / 'dinh_lu_tung_su_kien.csv', index=False, encoding='utf-8-sig')
from src.data.freeze import frozen_dir
flags = pd.read_csv(frozen_dir(cfg) / 'qc_flags.csv', parse_dates=['date']).date
me = [('persistence', 'PERS')] + [(m, 'E2') for m in M7 if m != 'persistence']
sens = C.sensitivity_excluding(pred, flags, me); print('Test:'); display(sens.round(3))
cvp = S.read_pred(cfg, 'cv', 1)                    # dự báo trên tập xác thực của 3 fold, mọi mô hình (results/du_bao.csv.gz)
rows = []
for m, e in me:
    for f in ['F1', 'F2', 'F3']:
        d = cvp[(cvp.model == m) & (cvp.exp == e) & (cvp.fold == f)]
        if not len(d): continue
        r = C.sensitivity_excluding(d.assign(model=m, exp=e), flags, [(m, e)]).iloc[0].to_dict(); r['fold'] = f; rows.append(r)
sens_cv = pd.DataFrame(rows); print('CV (val):'); display(sens_cv.round(3))
pd.concat({'test': sens, 'cv': sens_cv}).to_csv(OUT / 'do_nhay_102_ngay_nghi_nhieu.csv', encoding='utf-8-sig')

## 10. Chi phí và số tham số

In [ ]:
rows = []
for m in M7:
    c = conf[m]; rows.append({'mô hình': DISPLAY[m], 'siêu tham số': c.get('hp'), 'vòng (E2, test)': met[(met.model == m) & (met.exp.isin(['E2', 'PERS']))].n_iter.median()})
display(pd.DataFrame(rows).set_index('mô hình'))

## 11. Hình tổng hợp so sánh mô hình trên tập test (t+1, E2)

(a) NSE test của từng mô hình: mỗi điểm nhỏ là một hạt giống, vạch đứng là trung bình. (b) ΔNSE của LSTM và GRU so với từng
mô hình khác, kèm khoảng tin cậy 95 % bootstrap khối (`bootstrap_cau1.csv`); khoảng cắt qua 0 nghĩa là chưa có ý nghĩa thống kê.

In [ ]:
DLC, BLC, INK = '#2a78d6', '#8a8a85', '#3d3d3a'
fig, axs = plt.subplots(1, 2, figsize=(13, 4.2), gridspec_kw={'width_ratios': [1, 1.25]})
ax = axs[0]; names = [DISPLAY[m] for m in M7]
for i, m in enumerate(M7):
    d = met[(met.model == m) & (met.exp == ('PERS' if m == 'persistence' else 'E2'))].NSE.values
    col = DLC if m in ('lstm', 'gru') else BLC
    ax.scatter(d, np.full(len(d), i), s=22, color=col, alpha=.6, zorder=3)
    ax.plot([d.mean()] * 2, [i - .3, i + .3], color=col, lw=2.5, zorder=4)
    ax.text(d.mean(), i - .38, f'{d.mean():.3f}', fontsize=8, color=INK, ha='center')
ax.set_yticks(range(len(M7))); ax.set_yticklabels(names); ax.invert_yaxis(); ax.set_xlabel('NSE test (t+1, E2)')
ax.set_title('(a) NSE test theo mô hình', loc='left', fontsize=10)
ax = axs[1]
b = pd.read_csv(OUT / 'bootstrap_cau1.csv', encoding='utf-8-sig'); b = b[b['so sánh'].str.endswith('(E2)') & ~b['so sánh'].str.startswith('LSTM − GRU')]
lo = b['KTC 95 %'].str.extract(r'\[([-+0-9.]+); ([-+0-9.]+)\]').astype(float)
for i, (_, r) in enumerate(b.iterrows()):
    col = DLC if r['so sánh'].startswith('LSTM') else '#eb6834'
    ax.plot([lo.loc[r.name, 0], lo.loc[r.name, 1]], [i, i], color=col, lw=2)
    ax.scatter(r['ΔNSE (A − B)'], i, s=40, color=col, zorder=3)
ax.axvline(0, color=INK, lw=.8); ax.set_yticks(range(len(b))); ax.set_yticklabels(b['so sánh'].str.replace(' (E2)', '', regex=False), fontsize=8)
ax.invert_yaxis(); ax.set_xlabel('ΔNSE test (A − B), KTC 95 % bootstrap'); ax.set_title('(b) LSTM / GRU so với các mô hình khác', loc='left', fontsize=10)
plt.tight_layout(); plt.savefig(OUT / 'fig_so_sanh_test_H1.png', dpi=150); plt.show()

## 12. Vai trò dữ liệu Hủa Na theo mùa và trạng thái dòng chảy (t+1, dự báo đã lưu)

MAE của E1 … E4 tách theo **ngày đích**: mùa lũ (T7–T10) / mùa kiệt; dòng chảy quan trắc thấp (< Q50), trung bình (Q50–Q90),
cao (≥ Q90). Ngưỡng Q50 = 58 và Q90 = 173 m³/s lấy từ 2019–2023 (như notebook 01, cố định, không dùng dữ liệu test). Dự báo là
trung bình các hạt giống. Hai nguồn: **val của CV** (3 fold gộp — tập dùng để lựa chọn) và **test** (chỉ báo cáo). Chỉ dùng MAE
vì NSE trong nhóm con phụ thuộc mạnh vào phương sai của nhóm. Đây là phân tích mô tả, không kiểm định.

In [ ]:
daily_ = pd.read_csv(ROOT / cfg['paths']['processed'] / 'cuadat_huana_mua_daily.csv', parse_dates=['date'], index_col='date')
q_ = daily_.q_cd[(daily_.index <= '2023-12-31') & ~daily_.cd_missing]; Q50, Q90 = q_.quantile(.5), q_.quantile(.9)
print(f'Q50 = {Q50:.1f}, Q90 = {Q90:.1f} m³/s (2019–2023)')
def groups(dt, o):
    fl = dt.month.isin([7, 8, 9, 10])
    return {'cả năm': np.ones(len(o), bool), 'mùa lũ T7–T10': fl, 'mùa kiệt T11–T6': ~fl,
            f'Q thấp (< {Q50:.0f})': o < Q50, f'Q trung bình ({Q50:.0f}–{Q90:.0f})': (o >= Q50) & (o < Q90), f'Q cao (≥ {Q90:.0f})': o >= Q90}
rows = []
for src_, data in [('val CV (3 fold)', cvp[cvp.h == 1]), ('test', pred[pred.h == 1])]:
    for m in M7:
        if m == 'persistence': continue
        for e in ['E1', 'E2', 'E3', 'E4']:
            d = data[(data.model == m) & (data.exp == e) & data.dich_that.astype(bool)]
            keys = ['fold', 'ngay_goc'] if 'fold' in d else ['ngay_goc']
            a = d.groupby(keys).agg(o=('Q_do', 'first'), p=('Q_du_bao', 'mean')).reset_index()
            dt = pd.DatetimeIndex(a.ngay_goc) + pd.Timedelta(days=1)
            for g, mk in groups(dt, a.o.values).items():
                rows.append({'nguồn': src_, 'mô hình': DISPLAY[m], 'thực nghiệm': e, 'nhóm': g, 'n': int(mk.sum()),
                             'MAE': float(np.abs(a.p.values[mk] - a.o.values[mk]).mean())})
hs = pd.DataFrame(rows)
piv = hs.pivot_table(index=['nguồn', 'nhóm', 'mô hình'], columns='thực nghiệm', values='MAE')
for e in ['E2', 'E3', 'E4']: piv[f'Δ% {e}'] = 100 * (piv[e] / piv['E1'] - 1)
piv['n'] = hs[hs['thực nghiệm'] == 'E1'].set_index(['nguồn', 'nhóm', 'mô hình']).n
display(piv.round(1)); piv.to_csv(OUT / 'huana_theo_mua_trang_thai.csv', encoding='utf-8-sig')
print('Δ% MAE so với E1, trung vị 6 mô hình:')
med = piv.groupby(level=['nguồn', 'nhóm'], sort=False)[['Δ% E2', 'Δ% E3', 'Δ% E4']].median()
display(med.round(1)); med.to_csv(OUT / 'huana_theo_mua_trang_thai_trung_vi.csv', encoding='utf-8-sig')